# TxCoPilot for IgAN Reproducibility Notebook

**Dataset:** GSE37460 (microarray, GPL14663) IgAN vs healthy control  
**Reference:** Ventura, G. (2026). *TxCoPilot: A multi-agent AI system for causal network inference from transcriptomics data.* M.S. Thesis, Tufts University.

This notebook reproduces the IgAN case study from end to end:
1. **Agent 1**  DEG analysis (limma) and query parsing
2. **Agent 2**  TF activity, PROGENy, CARNIVAL ILP network inference
3. **Agent 3**  ReAct + RAG biological interpretation

Expected output: ~77 nodes · ~75 edges (minor variation due to floating-point ILP).

---
**Requirements:**
- Conda environment `txcopilot` (Python 3.11) with all packages from `requirements.txt`
- `ANTHROPIC_API_KEY` in a `.env` file at the repository root, or exported in the shell
- Data files in `data/`: `GSE37460_expression.csv` and `GSE37460_metadata.csv`

## 0. Setup

In [ ]:
import sys
import os
import logging
from pathlib import Path

# Repository root (one level up from notebooks/)
REPO_ROOT = Path().resolve().parent
AGENT1_DIR = REPO_ROOT / "Agent_1"
AGENT2_DIR = REPO_ROOT / "Agent_2"
AGENT3_DIR = REPO_ROOT / "Agent_3"
DATA_DIR   = REPO_ROOT / "data"

# Add agent packages to Python path
for p in [AGENT1_DIR, AGENT2_DIR, AGENT3_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"Repository root : {REPO_ROOT}")
print(f"Python version  : {sys.version.split()[0]}")

In [ ]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")
API_KEY = os.environ.get("ANTHROPIC_API_KEY")

assert API_KEY, (
    "ANTHROPIC_API_KEY not found. "
    "Add it to a .env file at the repo root or export it in your shell."
)
print("API key loaded.")

In [ ]:
# Optional: increase logging verbosity to follow pipeline progress
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(name)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

## 1. Data

GSE37460 (GPL14663) contains microarray gene expression profiles:
- **27 IgAN** samples (IgA Nephropathy kidney biopsies)
- **9 Control** samples (healthy kidney tissue)

Place the three files from `GSE_37460_formatted_files/` into a `data/` folder at the repository root:

```
TxCoPilot/
└── data/
    ├── GSE37460_expression.csv   # ID_REF × samples (log2 microarray intensities, Entrez IDs)
    ├── GSE37460_metadata.csv     # sample, condition  (IgAN / Control)
    └── GSE37460_DEGs.csv         # pre-computed limma DEGs: gene, log2FoldChange, pvalue, padj
```

**Note:** The expression matrix index column is `ID_REF` (Entrez numeric IDs). Agent 1's gene ID normalizer converts these to HGNC symbols automatically.

**Faster option:** `GSE37460_DEGs.csv` contains pre-computed limma results. Pass it as `precomputed_deg_path` in `UserQuery` (see the Agent 1 cell below) to skip the DEG analysis step entirely and go straight to CARNIVAL.

In [ ]:
import pandas as pd

EXPR_PATH = DATA_DIR / "GSE37460_expression.csv"
META_PATH = DATA_DIR / "GSE37460_metadata.csv"
DEGS_PATH = DATA_DIR / "GSE37460_DEGs.csv"   # pre-computed limma results

for p in [EXPR_PATH, META_PATH, DEGS_PATH]:
    assert p.exists(), f"Missing: {p}"

expr = pd.read_csv(EXPR_PATH, index_col=0)   # index = ID_REF (Entrez numeric IDs)
meta = pd.read_csv(META_PATH)
degs = pd.read_csv(DEGS_PATH)

print(f"Expression matrix : {expr.shape[0]:,} probes × {expr.shape[1]} samples")
print(f"Metadata          : {meta.shape[0]} samples")
print(f"Pre-computed DEGs : {len(degs):,} genes tested")
print(f"\nCondition counts:")
print(meta["condition"].value_counts().to_string())

In [ ]:
expr.iloc[:5, :4]  # Preview first 5 genes, 4 samples

## 2. Agent 1 Data Preparation & DEG Analysis

Agent 1:
1. Parses the natural-language query (LLM entity extraction)
2. Validates and QC-checks the expression matrix
3. Runs limma (via `inmoose`) for differential expression (microarray data) . (If this was bulk-RNA seq, it would run pyDESeq2)
4. Validates gene symbols against OmniPath
5. Returns an `Agent1Result` object for Agent 2

**Runtime:** ~1–3 minutes (includes two LLM calls and OmniPath validation).

In [ ]:
# Agent 1 uses relative paths for its output dir and gene cache.
# Running from AGENT1_DIR ensures those resolve correctly.
_orig_dir = os.getcwd()
os.chdir(AGENT1_DIR)

from agent1.agent1_main import Agent1Pipeline
from agent1.config import get_config
from agent1.schemas_agent1 import UserQuery

config = get_config()
config.anthropic_api_key = API_KEY
config.padj_threshold    = 0.05
config.log2fc_threshold  = 1.0

pipeline_a1 = Agent1Pipeline(config)

# ── Option A (default): run limma from expression matrix ──────────────────
# Agent 1 detects microarray data and runs inmoose/limma automatically.
query = UserQuery(
    query_text=(
        "What signaling pathways drive IgAN progression "
        "compared to healthy controls?"
    ),
    expression_path=str(EXPR_PATH),
    metadata_path=str(META_PATH),
    condition_column="condition",
    disease_label="IgAN",
    control_label="Control",
)

# ── Option B (faster): skip limma, use pre-computed DEGs ─────────────────
# Uncomment to use pre-computed results instead of re-running limma.
# query = UserQuery(
#     query_text=(
#         "What signaling pathways drive IgAN progression "
#         "compared to healthy controls?"
#     ),
#     precomputed_deg_path=str(DEGS_PATH),
#     expression_path=str(EXPR_PATH),
#     metadata_path=str(META_PATH),
#     condition_column="condition",
#     disease_label="IgAN",
#     control_label="Control",
# )

agent1_result = pipeline_a1.run(query)

os.chdir(_orig_dir)

print(f"Status : {agent1_result.status}")

In [ ]:
# Inspect Agent 1 outputs
deg = agent1_result.deg_results
qc  = agent1_result.quality_report
ent = agent1_result.extracted_entities

print("=== DEG Results ===")
print(f"  Comparison        : {deg.comparison}")
print(f"  Total genes tested: {deg.total_genes_tested:,}")
print(f"  Significant DEGs  : {deg.n_significant} "
      f"(↑ {deg.n_upregulated}  ↓ {deg.n_downregulated})")
print(f"  padj threshold    : {deg.padj_threshold}")
print(f"  |log2FC| threshold: {deg.log2fc_threshold}")

print("\n=== Quality Report ===")
print(f"  Data type   : {qc.data_type}")
print(f"  Samples     : {qc.n_samples}")
print(f"  Noise level : {qc.noise_level}")
print(f"  Quality tier: {qc.quality_tier}")

if qc.warnings:
    print(f"  Warnings    : {qc.warnings}")

print("\n=== Extracted Entities ===")
print(f"  Disease  : {ent.disease}")
print(f"  Pathways : {ent.pathways_mentioned}")
print(f"  Intent   : {ent.query_intent}")

In [ ]:
# Top DEGs
deg_df = pd.read_csv(deg.deg_file_path)
sig     = deg_df[deg_df["padj"] < 0.05].copy()
sig.sort_values("log2FoldChange", ascending=False, inplace=True)

print(f"Top 10 upregulated genes:")
print(sig.head(10)[["gene", "log2FoldChange", "padj"]].to_string(index=False))
print(f"\nTop 10 downregulated genes:")
print(sig.tail(10)[["gene", "log2FoldChange", "padj"]].to_string(index=False))

## 3. Agent 2 Causal Network Inference

Agent 2:
1. Infers TF activities from DEGs using DoRothEA (microarray) via decoupler VIPER
2. Scores pathway activities using PROGENy (MLM)
3. Selects CARNIVAL parameters via LLM + heuristic rules
4. Builds the Prior Knowledge Network from OmniPath (~70K edges)
5. Runs CARNIVAL ILP (CORNETO + HiGHS solver)
6. Returns an `Agent2Result` with a NetworkX DiGraph

**Runtime:** ~2–5 minutes (PKN download + ILP solve).  
**Note:** OmniPath is cached after the first download.

In [ ]:
# Agent 2 saves intermediate files relative to its own directory.
_orig_dir = os.getcwd()
os.chdir(AGENT2_DIR)

from agent2_main import Agent2Pipeline

pipeline_a2 = Agent2Pipeline(
    api_key=API_KEY,
    use_llm=True,   # set False to skip LLM parameter refinement (faster)
)

agent2_result = pipeline_a2.run(agent1_result)

os.chdir(_orig_dir)

G = agent2_result.carnival_network
print(f"Network: {G.number_of_nodes()} nodes  ·  {G.number_of_edges()} edges")

In [ ]:
import networkx as nx

# Node type breakdown
node_types = {}
for node, data in G.nodes(data=True):
    t = data.get("node_type", "unknown")
    node_types[t] = node_types.get(t, 0) + 1

print("Node types:")
for t, count in sorted(node_types.items(), key=lambda x: -x[1]):
    print(f"  {t:<20} {count}")

# Edge sign breakdown
activations = sum(1 for _, _, d in G.edges(data=True) if d.get("sign", 0) > 0)
inhibitions  = sum(1 for _, _, d in G.edges(data=True) if d.get("sign", 0) < 0)
print(f"\nEdge signs: {activations} activating  ·  {inhibitions} inhibiting")

# Top hub nodes by total degree
degree_series = pd.Series(dict(G.degree())).sort_values(ascending=False)
print(f"\nTop 10 hub nodes:")
print(degree_series.head(10).to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 10))

# Layout
pos = nx.spring_layout(G, seed=42, k=1.8)

# Color by node type
COLOR_MAP = {
    "transcription_factor": "#e06c75",
    "signaling": "#61afef",
    "perturbed": "#e5c07b",
    "unknown": "#abb2bf",
}
node_colors = [
    COLOR_MAP.get(G.nodes[n].get("node_type", "unknown"), "#abb2bf")
    for n in G.nodes()
]

# Edge colors: green = activation, red = inhibition
edge_colors = [
    "#98c379" if d.get("sign", 0) > 0 else "#e06c75"
    for _, _, d in G.edges(data=True)
]

nx.draw_networkx(
    G, pos=pos, ax=ax,
    node_color=node_colors,
    edge_color=edge_colors,
    node_size=400,
    font_size=7,
    arrows=True,
    arrowsize=12,
    width=1.2,
    alpha=0.9,
)

legend_handles = [
    mpatches.Patch(color=c, label=t.replace("_", " ").title())
    for t, c in COLOR_MAP.items()
]
ax.legend(handles=legend_handles, loc="upper left", fontsize=9)
ax.set_title(
    f"CARNIVAL Signaling Network — IgAN vs Control\n"
    f"{G.number_of_nodes()} nodes · {G.number_of_edges()} edges",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.savefig("igan_network.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to igan_network_notebook.png")

## 4. Agent 3 Biological Interpretation

Agent 3 uses a ReAct (Reason + Act) loop to:
1. Analyze the network using NetworkX tools (centrality, shortest paths, TF list)
2. Retrieve relevant passages from the RAG literature index (PubMedBERT + FAISS)
3. Compose a cited biological answer to the original question

**Runtime:** ~1–3 minutes (depends on number of ReAct iterations).  
**RAG index:** Pre-built at `Agent_3/rag_system/vector_db/` (53 papers, ~3,800 chunks).

In [ ]:
# Agent 3 loads the RAG index using Path(__file__).parent, which resolves
# correctly regardless of CWD, so no chdir needed here.
_orig_dir = os.getcwd()
os.chdir(AGENT3_DIR)

from agent3_react_loop import Agent3NetworkInterpreter, Agent3Config

config_a3 = Agent3Config(
    model="claude-sonnet-4-6",
    max_iterations=10,
    enable_rag=True,
    require_citations=True,
)

agent3 = Agent3NetworkInterpreter(api_key=API_KEY, config=config_a3)

question = query.query_text  # same question the user originally asked

response = agent3.answer_question(
    question=question,
    carnival_graph=G,
    query_type=agent1_result.extracted_entities.query_intent or "all",
    disease_summary=agent2_result.summary,
)

os.chdir(_orig_dir)

print(f"ReAct iterations : {response.get('iterations', '?')}")
print(f"Citations found  : {len(response.get('citations', []))}")

In [ ]:
# Print the final answer
print(response["answer"])

In [ ]:
# Print citations
print("=== Supporting Literature ===")
for i, cite in enumerate(response.get("citations", []), 1):
    source = cite.get("source", "").replace(".pdf", "")
    score  = cite.get("score", 0)
    print(f"  [{i}] {source}  (relevance: {score:.2f})")

## 5. Expected Results

| Parameter | Value |
|---|---|
| Dataset | GSE37460 (microarray, GPL14663) |
| Comparison | IgAN vs Control |
| DEG method | limma (inmoose) |
| padj threshold | 0.05 |
| \|log2FC\| threshold | 1.0 |
| CARNIVAL beta | 0.2 (LLM-selected for low-noise microarray) |
| PKN | OmniPath full (~70,274 edges) |
| **Expected nodes** | **~77** |
| **Expected edges** | **~75** |
| **Edge/node ratio** | **~0.97** (CARNIVAL ILP tree constraint) |

> **Note on reproducibility:** The HIGHS ILP solver is deterministic given identical input, but floating-point differences across hardware/OS may occasionally yield slightly different CARNIVAL solutions. The node and edge counts above are typical; however expect variation.

### Key pathways expected in the network

Based on IgAN literature:
- TGF-β / SMAD signaling
- MAPK / ERK cascade (IgA1 immune complex activation)
- NF-κB (mesangial inflammation)
- Wnt / β-catenin (complement and podocyte dysfunction)
- PI3K / Akt

---

### Full interactive results

For the Cytoscape network visualization, follow-up chat, and KEGG enrichment panel, launch the Streamlit UI:

```bash
cd Agent_1
streamlit run streamlit_app.py
```